# Can Machine Learning Improve Technical-Rule Trade Selection?

## A Walk-Forward Study of U.S. Equities

### Abstract

This study examines whether machine learning models can improve the selection of trades generated by six technical analysis rules: breakout, pullback, reversal, moving average golden cross, stochastic crossover and MACD crossover.

Daily U.S. equity data from January 2016 to October 2026 are used to contruct a liquidity filtered universe of common stocks and ADRs. Each signal is evaluated using the same pre-defined triple barrier exit rule, transaction cost assumptions and maximum holding period. LightGBM and logistic regression are trained using yearly walk-forward validation with a 21 trading day embargo.

Across all out of sample years, LightGBM achieved an AUC of 0.512, compared with 0.530 for logistic regression. In the post October 2024, higher confidence data period, the corresponding AUC values were 0.536 and 0.511. However, probability decile performance was not monotonic, and permutation importance indicated the most of LightGBM's predictive ability came from market return and volatility features. Removing these market regime variables reduced clean period AUC to approximately 0.50.

This result provide limited evidence that machine learning can identify favourable market environments, but weak evidence of stable stock level selection ability. Thses findings should be treated as exploratory rather than evidence of a deployable trading strategy.


### Research Question

Can a machine learning model rank technical rule trade candidates more effectively that the underlying rules alone after estimated transaction costs?

### Study Design

This experiment follows five stages:

1. Construct a daily tradable universe using price and liquidity filters.
2. Generate candidate trades from six deterministic technical rules.
3. Simulate every candidate with the same entry, exit, and cost assumptions.
4. Train LightGBM and logistic regression models using only information available on the signal date.
5. Evaluate discrimination, calibration and trade-ranking performance using walk-forward out of sample predictions. 

## 1. Experimental Configuration

- TOP_N, MIN_PRICE, MIN_VOL: who is allowed into the tradable universe each day.
- MAX_HOLD: the time barrier. A trade is close after this many trading days no matter what.
- STOP_ATR, TARGET_ATR: the stop sits 2 ATR below entry, the target 3 ATR above. These are deliberately not optimised. Change them only before looking at any results.
- CLEAN_START: first date with delisted stocks in your database.


In [2]:
import numpy as np
import pandas as pd

import N_DB

#default length 80
pd.set_option('display.width', 160)

START = '2016-01-01'
TOP_N = 1500        # tradable universe: the N most liquid stocks each day
MIN_PRICE = 5
MIN_DVOL = 2e6      # 20-day median dollar volume, USD

MAX_HOLD = 20       # time barrier, trading days
STOP_ATR = 2.0      # stop-loss distance below entry, in ATRs
TARGET_ATR = 3.0    # profit-target distance above entry, in ATRs

CLEAN_START = pd.Timestamp('2024-10-01')
TEST_YEARS = range(2019, 2027)


## 2. Data and Universe Construction

Load all the open, high, low, close, date and volume

In [3]:
bars = N_DB.query(f"""
            SELECT symbol, open, high, low, close, date, volume
            FROM bars_daily
            WHERE date >= '{START}'                 
        """)

bars['date'] = pd.to_datetime(bars['date'])

print(f"{len(bars):,} rows, "
      f"\n {bars['symbol'].nunique():,} symbols, "
      f"\n {bars['date'].min().date()} -> {bars['date'].max().date()}")

19,478,795 rows, 
 16,025 symbols, 
 2016-01-04 -> 2026-10-14


ETFs, warrents, units, and preferred shares behave differently and would pollute both the rules and the labels.

In [4]:
ref = N_DB.query("SELECT ticker, type, delisted FROM tickers_ref")
# print(ref)

types = (ref.dropna(subset=['type'])
            .sort_values('delisted')
            .drop_duplicates('ticker' ,keep='last')
            .set_index('ticker')['type'])

uni = N_DB.query("""
            SELECT DISTINCT ON (symbol) symbol, security_type
            FROM universe 
            ORDER BY symbol, last_seen DESC         
        """).set_index('symbol')['security_type']

# print(uni)
uni_type = uni.map({'common': 'CS', 'adr': 'ADRC'})

syms = pd.Series(bars['symbol'].unique())
stype = syms.map(types).fillna(syms.map(uni_type))
keep = set(syms[stype.isin(['CS', 'ADRC'])])

bars = bars[bars['symbol'].isin(keep)]
print(f"Common + ADR: {len(keep):,} symbols, {len(bars):,} rows")

Common + ADR: 6,977 symbols, 10,870,774 rows


One table per field: rows = dates, columns = symbols. Everything downstream is built from these five.

Also compute here the handful of indicators that both the entry rules and the exit rule need: 

- ret: daily return
- atr: Average True Range, Wilder-smoothed over 14 days. The single most important number in this notebook - stops, targets, and most features are measuared in ATRs.
- ma20, ma50 : moving averages used to define 'uptrend'.
- rsi2: a 2 day RSI. Below 10 means the stock has just had a sharp two-day drop. 

In [5]:
def wide(col):
    return (bars.pivot(index='date', columns='symbol', values= col).sort_index().astype('float32'))

open_ = wide('open')
close = wide('close')
high = wide('high')
low = wide('low')
vol = wide('volume')
dvol = close * vol
del bars

T, N = close.shape
print(f"{T:,} trading days x {N:,} symbols")

ret = close.pct_change(fill_method= None)

# ATR (price range): true range = max(high-low, |high-prev close|, |low-prev close|), Wilder-smoothed
pc = close.shift(1)
tr = np.fmax(np.fmax(high - low, (high - pc).abs()), (low - pc).abs())
atr = tr.ewm(alpha= 1/14, adjust= False, min_periods= 14).mean()
atr_pct = atr / close

ma20 = close.rolling(20, min_periods= 15).mean()
ma50 = close.rolling(50, min_periods= 40).mean()
avgvol20 = vol.rolling(20, min_periods= 15).mean()

def rsi(n):
    d = close.diff()
    g = d.clip(lower= 0).ewm(alpha= 1/n, adjust= False, min_periods= n).mean()
    l = (-d.clip(upper= 0)).ewm(alpha= 1/n, adjust= False, min_periods= n).mean()
    return 100 - 100 / (1 + g/l)

rsi2 = rsi(2)

del pc, tr


2,714 trading days x 6,977 symbols


Top 1500 by liquidity, price >= $5, median dollar volume >= $2M

In [6]:
liq = dvol.rolling(20, min_periods=15).median()
liq_rank = liq.rank(axis= 1, ascending= False)

univ = (liq_rank <= TOP_N) & (close >= MIN_PRICE) & (liq >= MIN_DVOL) & atr.notna()
mkt = ret.where(univ).mean(axis= 1)

print("Average universe size per year: ")
print(univ.sum(axis= 1).resample('YE').mean().round().astype(int).to_string())


Average universe size per year: 
date
2016-12-31    1379
2017-12-31    1474
2018-12-31    1480
2019-12-31    1472
2020-12-31    1464
2021-12-31    1486
2022-12-31    1473
2023-12-31    1477
2024-12-31    1472
2025-12-31    1472
2026-12-31    1457
Freq: YE-DEC


## 3. Candidate Signal Generation

Three simple rules, each evaluated at the close of day t, each firing only on days when something actually happened:

|Rule | Fires when | The idea|
|--|--|--|
|breakout| Close above the previous 60 day high and volume > 1.5x its 20 day average| Buying strength with confirmation|
|pullback| Uptrend (close > MA50, MA20 > MA50) and close within 1 ATR of MA20 and RSI(2) < 10 | Buying a short, sharp dip in a trend|
|reversal| Today's low broke below the previous 20 day low, but the close recovered above it, on above-average volume| A failed breakdown; sellers exhausted|
|Golden Cross | MA50 crosses above MA200 | Captures the beginning of a potential long-term bullish trend.|
| KD Cross | The stochastic %K line crosses above %D while %K is below 30 | Identifies improving momentum from an oversold or weak-price region.|
|MACD Cross | MACD crosses above its signal line while MACD remains below zero| Identifies an early bullish momentum shift before the longer trem trend becomes positive.|

Each rule is a yes/no table (dates x symbols). Keep only the fires that are inside the universe, and drop signals in the last MAX_HOLD + 2 days because those trades cannot finish yet.

Why rules, not a model, for entries? The model will be asked "is this breakout real?", a question where price and volume genuinely carry information.

One rule per stock per day. If two rules fire together, only keep the first in priority order. One trade per stock per holding window. If the same stock fires again within 20 days skip it.

In [38]:
hi60_prev = high.rolling(60, min_periods= 50).max().shift(1)
lo20_prev = low.rolling(20, min_periods= 15).min().shift(1)
uptrend = (close > ma50) & (ma20 > ma50)
ma200 = close.rolling(200, min_periods= 180).mean()

# KD (slow stochastic, 9-3-3)
lo9, hi9 = low.rolling(9, min_periods=7).min(), high.rolling(9, min_periods=7).max()
kd_k = (100 * (close - lo9) / (hi9 - lo9).replace(0, np.nan)).rolling(3, min_periods=2).mean()
kd_d = kd_k.rolling(3, min_periods=2).mean()

# MACD (12, 26, 9)
macd   = close.ewm(span=12, adjust=False).mean() - close.ewm(span=26, adjust=False).mean()
signal = macd.ewm(span=9, adjust=False).mean()

rules = {
    'breakout': (close > hi60_prev) & (vol > 1.5 * avgvol20),
    'pullback': uptrend & ((close - ma20).abs() <= atr) & (rsi2 < 10),
    'reversal': (low < lo20_prev) & (close > lo20_prev) & (vol > avgvol20),
    'golden_cross': (ma50 > ma200) & (ma50.shift(1) <= ma200.shift(1)) & univ,
    'kd_cross': (kd_k > kd_d) & (kd_k.shift(1) <= kd_d.shift(1)) & (kd_k < 30),
    'macd_cross': (macd > signal) & (macd.shift(1) <= signal.shift(1)) & (macd < 0),
}

last_ok = T - (MAX_HOLD + 2)    # signals this close to the end cannot finish a trade

frames = []
for name, cond in rules.items():
    c = (cond & univ).iloc[:last_ok]
    ii, jj = np.nonzero(c.values)
    idx = pd.MultiIndex.from_arrays([c.index[ii], c.columns[jj]], names=['date', 'symbol'])
    frames.append(pd.DataFrame({'rule': name}, index= idx))

cand = pd.concat(frames)
cand = cand[~cand.index.duplicated(keep= 'first')].sort_index()     # one rule per stock-day

# one trade per stock per holding window
pos_d = pd.Series(np.arange(T), index= close.index)    # date   -> row number
pos_s = pd.Series(np.arange(N), index= close.columns)  # symbol -> column number
cand['i0'] = pos_d[cand.index.get_level_values('date')].values
cand['j'] = pos_s[cand.index.get_level_values('symbol')].values

def thin(positions, gap):
    keep, last = [], -10**9
    for p in positions:
        ok = (p - last) >= gap
        keep.append(ok)
        if ok:
            last = p
    return keep

cand['keep'] = cand.groupby(level= 'symbol')['i0'].transform(lambda s: thin(s.values, MAX_HOLD))
print(f'Before thinning: {len(cand):,}, after: {cand['keep'].sum():,}')
cand = cand[cand['keep']].drop(columns= 'keep')

print('\n Candidates per rule: ')
print(cand['rule'].value_counts().to_string())
print('\n Candidates per year: ')
print(cand.groupby(cand.index.get_level_values('date').year).size().to_string())


Before thinning: 486,793, after: 142,028

 Candidates per rule: 
rule
kd_cross        46041
pullback        32776
reversal        24644
macd_cross      17559
breakout        17173
golden_cross     3835

 Candidates per year: 
date
2016    12026
2017    12891
2018    13878
2019    12413
2020    13136
2021    13488
2022    13794
2023    13274
2024    13937
2025    13466
2026     9725


## 4. Trade Simulation and Target Definition

Every candidate is traded the same way, decided before we look at any result:

1. Enter at the next day's open.
2. Stop loss at entry - 2 ATR. Profit target at entry + 3 ATR. (ATR as of the signal day.)
3. Each day: if the open gaps through a barrier, exit at the open. Otherwise if the day's low touches the stop, exit at the stop; 
            if the high touches the target, exit at the target.
            if both happen on the same day we assume the stop was hit first - the conservative choice.
4. After MAX_HOLD days with neither barrier hit, exit at the day's close.
5. If the stocks stops trading, exit at its last available close.

Costs are charged per side and depend on how liquid the stock is on the signal day.

Each trade gets: entry, exit, days, why(target/ stop/ time/ delist), gross, net and R.

R is the number to watch. R = net return / (stop distance as a % of entry). A full stop-out is about -1R, a clean target hit about +1.5R. Measuring in R makes a $5 biotech and a $500 blue chip comparable.

In [39]:
O, H, L, C, A = open_.values, high.values, low.values, close.values, atr.values

def cost_of(rank):
    return np.where(rank <= 500, 0.0005, np.where(rank <= 1000, 0.0010, 0.0025))

def simulate(i0, j, cost):
    '''One trade. i0 = signal-day row, j = symbol column. Returns a dict or None.'''
    i = i0 + 1
    if np.isnan(C[i,j]):
        return None
    entry = O[i, j] if not np.isnan(O[i,j]) else C[i,j]
    a = A[i0, j]
    if np.isnan(entry) or np.isnan(a) or a <= 0 or entry <= 0:
        return None
    
    stop, target = entry - STOP_ATR * a, entry + TARGET_ATR * a
    k, exit_px, why = i, np.nan, 'time'
    while k < i + MAX_HOLD and k < T:
        c = C[k, j]
        if np.isnan(c):
            k -= 1 
            exit_px, why = C[k, j], 'delist'
            break
        
        o = O[k, j] if not np.isnan(O[k,j]) else c
        h = H[k, j] if not np.isnan(H[k,j]) else c
        l = L[k, j] if not np.isnan(L[k,j]) else c
        if k > i and o <= stop:
            exit_px, why = o, 'stop'
            break
        if k > i and o >= target:
            exit_px, why = o, 'target'
            break
        if l <= stop:
            exit_px, why = stop, 'stop'
            break
        if h >= target:
            exit_px, why = target, 'target'
            break
        
        k += 1
        
    if why == 'time':
        k -= 1
        exit_px = C[k, j]   
        
    gross = exit_px / entry - 1
    net = gross - 2 * cost
    return dict(entry= entry, exit= exit_px, days= k - i + 1, why= why,
                gross= gross, net= net, R= net / (STOP_ATR * a / entry))
    
cand['liq_rank'] = liq_rank.values[cand['i0'].values, cand['j'].values]
cand['cost'] = cost_of(cand['liq_rank'].values)

out = [simulate(i0, j, c) for i0, j, c in zip(cand['i0'], cand['j'], cand['cost'])]
trades = pd.DataFrame([o if o else {} for o in out], index= cand.index)
cand = cand.join(trades).dropna(subset= ['R'])

print(f"{len(cand):,} simulate trades\n")
print("How trades ended (share of trades): ")
print(pd.crosstab(cand['rule'], cand['why'], normalize= 'index').round(3))
print(cand.groupby('rule').agg(n= ('R', 'size'), win_rate= ('R', lambda s: (s>0).mean()),
                               avg_R= ('R', 'mean'), avg_net= ('net', 'mean'),
                               avg_days= ('days', 'mean')).round(3))    
    


141,891 simulate trades

How trades ended (share of trades): 
why           delist   stop  target   time
rule                                      
breakout       0.009  0.480   0.300  0.210
golden_cross   0.013  0.472   0.300  0.214
kd_cross       0.013  0.429   0.294  0.263
macd_cross     0.010  0.430   0.302  0.258
pullback       0.012  0.452   0.323  0.213
reversal       0.011  0.427   0.298  0.263
                  n  win_rate  avg_R  avg_net  avg_days
rule                                                   
breakout      17141     0.441 -0.029   -0.001    10.615
golden_cross   3834     0.445 -0.020   -0.001    11.227
kd_cross      46007     0.469  0.033    0.003    12.059
macd_cross    17542     0.474  0.047    0.004    12.194
pullback      32751     0.463  0.034    0.002    11.244
reversal      24616     0.475  0.043    0.004    11.777


win = 1 if the trade's net R is positive, else 0. One number per trade. This is a classification problem now: the model outputs a probability, which we will turn into an expected value later.

Keep R as well, because "how much" matters for the portfolio even though the model learns "whether".

In [40]:
cand['win'] = (cand['R'] > 0).astype(int)
print(f"Overall win rate: {cand['win'].mean():.3f}")
print(cand.groupby(cand.index.get_level_values('date').year)['win'].mean().round(3).to_string())

Overall win rate: 0.465
date
2016    0.497
2017    0.528
2018    0.409
2019    0.522
2020    0.510
2021    0.446
2022    0.392
2023    0.441
2024    0.476
2025    0.469
2026    0.435


## 5. Feature Engineering

|Group | Feature | What it says | Normalisation|
|--|--|--|--|
| Trend | dist_ma20 | How far price is above | below the 20 days average, in ATRs| ATR units|
| Trend | ma20_slope | Change in MA20 over 10 days, in ATRs | ATR units |
| Trend | adx_rk | Trend strength (ADX 14)| cross-sectional rank|
| Position | pos60 | where price sits in its 60-day range (0 = low, 1= high)|already 0 - 1|
| Position | clv | Where today's close sits in today's range | already 0 - 1|
| Position | days_hi252_rk | Days since the 52 week high | rank |
| Event | vol_ratio_rk | Today's volume vs 20 day average | rank |
| Relative strength | r20_sec, r60_sec | 20/60 day return ranked within sector on that day | rank within sector|
| Volatility | atr_pct_rk | ATR as % of price | rank |
| Volatility | squeeze_rk | 20 day range vs its 120 day average (low = coiled)| rank |
| Liquidity | log_dvol_rk | Dollar volume (a size proxy) | rank |
| Regime | mkt_ret_20, mkt_vol_20 | Market's recent return and volatility | raw |
| Rule | is_breakout, is_pullback | Which rule fired (reversal= both 0) | 0/1 |



In [41]:
def days_since(cond):
    '''Trading days since cond was last True (never = NaN).'''
    c = cond.values
    out = np.full(c.shape, np.nan, dtype= 'float32')
    last = np.full(c.shape[1], np.nan)
    for i in range(len(c)):
        last = np.where(c[i], i, last)
        out[i] = i - last
    return pd.DataFrame(out, index= cond.index, columns= cond.columns)

# ADX (Wilder)
up, dn = high.diff(), -low.diff()
plus_dm = up.where((up > dn) & (up > 0), 0)
minus_dm = dn.where((dn > up) & (dn > 0), 0)
w = dict(alpha= 1/14, adjust= False, min_periods= 14)
pdi = 100 * plus_dm.ewm(**w).mean() / atr
mdi = 100 * minus_dm.ewm(**w).mean() / atr
adx = (100 * (pdi - mdi).abs() / (pdi + mdi)).ewm(**w).mean()
del up, dn, plus_dm, minus_dm, pdi, mdi

hi60 = high.rolling(60, min_periods=50).max()
lo60 = low.rolling(60, min_periods=50).min()
rng = (high - low).replace(0, np.nan)
rng20 = (high.rolling(20, min_periods=15).max() - low.rolling(20, min_periods=15).min()) / close

# already comparable across stocks
RAW_FEATS = {
    'dist_ma20': (close - ma20) / atr,
    'dist_ma50': (close - ma50) / atr,
    'ma20_slope': (ma20 - ma20.shift(10)) / atr,
    'pos60': (close - lo60) / (hi60 - lo60),
    'clv': (close - low) / rng,
}

# will be replaced by their per-date percentile rank
RANK_FEATS = {
    'adx': adx,
    'days_hi252': days_since(close >= close.rolling(252, min_periods=200).max()),
    'vol_ratio': vol / avgvol20,
    'r20': close / close.shift(20) - 1,
    'r60': close / close.shift(60) - 1,
    'atr_pct': atr_pct,
    'squeeze': rng20 / rng20.rolling(120, min_periods= 100).mean(),
    'log_dvol': np.log(liq.where(liq > 0)),
}

del hi60, lo60, rng, rng20, adx


Now turn the wide tables into one long table F with one row per (date, symbol) for every universe stock on every day - not just candidates. That is necessary because a cross-sectional rank only means something if it is taken over the whole universe.

In [ ]:
feats = {}

for name, df in {**RAW_FEATS, **RANK_FEATS}.items():
    feats[name] = df.where(univ).stack().dropna().astype('float32')
    
F = pd.concat(feats, axis= 1)
F.index.names = ['date', 'symbol']
del feats, RAW_FEATS, RANK_FEATS
print(f"Universe rows: {len(F):,}")

# sector = first 2 digits of SIC code
sec = N_DB.query("""
        SELECT DISTINCT ON (symbol) symbol, sic
        FROM universe
        ORDER BY symbol, last_seen DESC                 
        """)

sector = (sec.dropna(subset=['sic'])
            .set_index('symbol')['sic']
            .astype(int)
            .astype(str).str.zfill(4).str[:2])

F['sector'] = sector.reindex(F.index.get_level_values('symbol')).fillna('00').values

# percentile ranks across the universe on each date
for c in ['adx', 'days_hi252', 'vol_ratio', 'r20', 'r60', 'atr_pct', 'squeeze', 'log_dvol']:
    F[c + '_rk'] = F.groupby(level= 'date')[c].rank(pct=True).astype('float32')


# relative strength: rank within sector on each date
for c in ['r20', 'r60']:
    F[c + '_sec'] = (F.groupby([pd.Grouper(level='date'), 'sector'])[c]
                     .rank(pct= True).astype('float32'))
    
F = F.drop(columns= ['adx', 'days_hi252', 'vol_ratio', 'r20', 'r60', 'atr_pct', 'squeeze', 'log_dvol'])

# market regime, same value for every stock on a date
mkt_idx = (1 + mkt.fillna(0)).cumprod()
d = F.index.get_level_values('date')
F['mkt_ret_20'] = (mkt_idx / mkt_idx.shift(20) - 1).reindex(d).values.astype('float32')
F['mkt_vol_20'] = mkt.rolling(20).std().reindex(d).values.astype('float32')

# attach features to the candidate trades
data = cand.join(F, how='inner')
data['is_breakout'] = (data['rule'] == 'breakout').astype(int)
data['is_pullback'] = (data['rule'] == 'pullback').astype(int)

FEATURES = ['dist_ma20', 'ma20_slope', 'clv',
            'adx_rk', 'days_hi252_rk', 'vol_ratio_rk',
            'r20_sec', 'r60_sec',
            'atr_pct_rk', 'squeeze_rk', 'log_dvol_rk',
            'mkt_ret_20', 'mkt_vol_20', 'is_breakout', 'is_pullback']

SPARES   = ['dist_ma50', 'r20_rk', 'r60_rk']     # computed, kept in `data`, not used by default

print(f"{len(data):,} trades x {len(FEATURES)} features")
print("\n Missing share per features: ")
print(data[FEATURES].isna().mean().sort_values(ascending= False).head(6).round(3).to_string())

Universe rows: 3,973,933
141,891 trades x 15 features

 Missing share per features: 
days_hi252_rk    0.146
squeeze_rk       0.047
mkt_vol_20       0.028
r60_sec          0.023
adx_rk           0.011
ma20_slope       0.011


### Check for redundant features

Two features with a Spearman correlation above 0.8 are telling the model the same thing, that does not hurt a tree model much, but it muddies the importance table and hurts the logistic baseline. If pairs show up here, drop one of each pair from FEATURES and re-run from this cell.

In [43]:
corr = data[FEATURES].corr(method= 'spearman')
pairs = (corr.where(np.triu(np.ones(corr.shape, bool), k= 1))
            .stack().rename('rho'))

print(pairs[pairs.abs() > 0.8].round(2).to_string() if (pairs.abs() > 0.8).any()
                                                    else "No feature pairs with |rho| > 0.8")

print("\n Spares vs the feature they would replace: ")
print(data[FEATURES + SPARES].corr(method= 'spearman').loc[SPARES, ['dist_ma20', 'r20_sec', 'r60_sec']].round(2))

No feature pairs with |rho| > 0.8

 Spares vs the feature they would replace: 
           dist_ma20  r20_sec  r60_sec
dist_ma50       0.80     0.61     0.59
r20_rk          0.67     0.85     0.49
r60_rk          0.46     0.48     0.85


## 6. Walk-Forward Validation

For each test year Y:
train : all trades with signal date <= (first test date - 21 trading days)
validate : the last year inside train (used only to decide when to stop adding trees)
test : trades with signal date in year Y

The 21 day embargo exists because a trade signalled on 20 December is still open in January. Its label uses January prices. Without the gap, the model would be trained on outcomes that overlap the test period.

Two models:
- LightGBM classifier, kept small on purpose: 15 leaves, at least 200 trades per leaf, strong L2 penalty, and early stopping on the validation year instead of a fixed number of trees.
- Logistic regression as the honest baseline. If LightGBM does not beat it by a clear margin, the non-linear structure is not there and the simpler models wins.

Also compute an expected value in R for every test trade:

```
EV = p * (average R of winning trades in training) + (1 - p) * (average R of losing trades in training)
```

A trade is worth taking only if EV > 0. Both averages come from the training years, never the test year.

In [44]:
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

dates = data.index.get_level_values('date')
years = dates.year
EMBARGO = MAX_HOLD + 1

LGB = dict(n_estimators= 2000, learning_rate= 0.02, num_leaves= 15, min_child_samples= 200,
           subsample= 0.8, subsample_freq= 1, colsample_bytree= 0.8, reg_lambda= 5.0,
           metric= 'auc', verbose= -1)

for c in ['p_lgb', 'p_log', 'ev']:
    data[c] = np.nan
    
models = {}

for year in TEST_YEARS:
    test = (years == year)
    if not test.any():
        continue
    
    first = dates[test].min()
    cut = close.index[pos_d[first] - EMBARGO]
    train = (dates <= cut)
    valid = train & (years == year - 1)
    fit = train & ~valid

    m = lgb.LGBMClassifier(**LGB)
    m.fit(data.loc[fit, FEATURES], data.loc[fit, 'win'],
          eval_X = data.loc[valid, FEATURES],
          eval_y = data.loc[valid, 'win'],
          callbacks=[lgb.early_stopping(50, verbose= False)])
    data.loc[test, 'p_lgb'] = m.predict_proba(data.loc[test, FEATURES]) [:, 1]
    models[year] = m

    lr = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                       LogisticRegression(C=0.1, max_iter=1000))
    lr.fit(data.loc[train, FEATURES], data.loc[train, 'win'])
    data.loc[test, 'p_log'] = lr.predict_proba(data.loc[test, FEATURES]) [:, 1]
    
    r_win = data.loc[train & (data['win'] == 1), 'R'].mean()
    r_loss = data.loc[train & (data['win'] == 0), 'R'].mean()
    
    p = data.loc[test, 'p_lgb']
    data.loc[test, 'ev'] = p * r_win + (1 - p) * r_loss

    print(f"{year}: fit {fit.sum():>7,} valid {valid.sum():>6,} test {test.sum():>6,} "
          f"tress {m.best_iteration_:>4} r_win {r_win:+.2f} r_loss {r_loss:+.2f}")


2019: fit  24,916 valid 12,520 test 12,413 tress    1 r_win +1.17 r_loss -0.99
2020: fit  38,794 valid 11,456 test 13,136 tress    2 r_win +1.17 r_loss -0.99
2021: fit  51,207 valid 12,129 test 13,488 tress    2 r_win +1.17 r_loss -0.99
2022: fit  64,343 valid 12,690 test 13,794 tress   42 r_win +1.18 r_loss -0.98
2023: fit  77,831 valid 12,500 test 13,274 tress   16 r_win +1.17 r_loss -0.98
2024: fit  91,625 valid 12,317 test 13,937 tress    2 r_win +1.18 r_loss -0.98
2025: fit 104,899 valid 12,619 test 13,464 tress  258 r_win +1.19 r_loss -0.98
2026: fit 118,836 valid 12,368 test  9,591 tress    1 r_win +1.19 r_loss -0.98


## 7. Out-of-Sample Results

Four views, each answering a different question.

7a. AUC by year. Can the model tell winners from losers at all? Look for values consistently above 0.5, and compare LightGBM with the logistic baseline. Then look at the clean 2024-10+ row: that is the only period free of survivorship bias. 

In [45]:
scored = data['p_lgb'].notna()

def auc_block(mask):
    sub = data[mask & scored]
    if sub['win'].nunique() < 2 or len(sub) < 50:
        return pd.Series({'n': len(sub), 'AUC lgb': np.nan, 'AUC logistic': np.nan})
    return pd.Series({'n': len(sub),
                        'AUC lgb': roc_auc_score(sub['win'], sub['p_lgb']),
                        'AUC logistic': roc_auc_score(sub['win'], sub['p_log'])})
        
rows = {str(y): auc_block(years==y) for y in TEST_YEARS if (years == y).any()}
rows['all test years'] = auc_block(years >= TEST_YEARS[0])
rows['clean 2024-10+'] = auc_block(dates >= CLEAN_START)
print(pd.DataFrame(rows).T.round(3).to_string())

                       n  AUC lgb  AUC logistic
2019             12413.0    0.567         0.572
2020             13136.0    0.507         0.568
2021             13488.0    0.561         0.559
2022             13794.0    0.516         0.561
2023             13274.0    0.516         0.529
2024             13937.0    0.566         0.542
2025             13464.0    0.545         0.518
2026              9591.0    0.553         0.505
all test years  103097.0    0.512         0.530
clean 2024-10+   26787.0    0.536         0.511


7b. Calibration and payoff by probability decile. Sort test trades into ten buckets by predicted probability. A useful model shows win_rate rising from bucket 1 to 10 and avg_R rising with it. If p_pred and win_rate are far apart, the probabilities are miscalibrated and the EV rule in Step 11 will be off - but the ordering can still be useful.

In [46]:
t = data[scored].copy()
t['bin'] = pd.qcut(t['p_lgb'].rank(method= 'first'), 10, labels= False) + 1
print(t.groupby('bin').agg(n=('win', 'size'), p_pred= ('p_lgb', 'mean'), win_rate= ('win', 'mean'),
                           avg_R= ('R', 'mean'), avg_net= ('net', 'mean'),
                           avg_days= ('days', 'mean')).round(3).to_string())

         n  p_pred  win_rate  avg_R  avg_net  avg_days
bin                                                   
1    10310   0.428     0.439 -0.023   -0.002    11.367
2    10310   0.467     0.438 -0.039   -0.002    10.687
3    10309   0.469     0.455  0.025    0.002    10.860
4    10310   0.473     0.513  0.158    0.012    11.866
5    10310   0.479     0.432 -0.031   -0.002    11.305
6    10309   0.487     0.419 -0.072   -0.000    11.745
7    10310   0.491     0.468  0.041    0.007    11.577
8    10309   0.504     0.472  0.027    0.002    11.917
9    10310   0.516     0.557  0.223    0.012    12.710
10   10310   0.566     0.421 -0.076   -0.005    12.129


7c. Model picks vs the rest, per rule. For each rule, split its trades into the model's top 30% by probability and the remaining 70%. Lift is the difference in average R; t is a Welch t-statistic. This is the single most important table in the notebook:

- lift > 0 with t > 2 in the clean period: the model adds somthing on top of the rule.
- lift ~ 0: the rule is all there is; the model is noise. Drop the model, keep or drop the rule.
- lift > 0 only before 2024-10: probably survivorship bias; do not trust it.

In [47]:
from scipy.stats import ttest_ind

def top_vs_rest(sub, col= 'p_lgb', q= 0.7):
    thr = sub[col].quantile(q)
    top = sub[sub[col] >= thr]
    rest = sub[sub[col] < thr]
    tstat = ttest_ind(top['R'], rest['R'], equal_var=False).statistic if len(top) > 30 else np.nan
    return pd.Series({'n': len(sub), 'R_rule': sub['R'].mean(), 'R_top30': top['R'].mean(),
                      'R_rest': rest['R'].mean(), 'lift': top['R'].mean() - rest['R'].mean(), 't': tstat})
    
rows= {}
for period, mask in [('test 2019+', years >= TEST_YEARS[0]), ('clean 2024-10+', dates >= CLEAN_START)]:
    for rule in rules:
        sub = data[scored & mask & (data['rule'] == rule)]
        if len(sub) > 100:
            rows[(period, rule)] = top_vs_rest(sub)
    rows[(period, 'ALL')] = top_vs_rest(data[scored & mask])
print(pd.DataFrame(rows).T.round(3).to_string())



                                    n  R_rule  R_top30  R_rest   lift       t
test 2019+     breakout       11934.0  -0.022   -0.095   0.011 -0.106  -4.635
               pullback       23789.0   0.031    0.010   0.040 -0.029  -1.716
               reversal       17891.0   0.043    0.086   0.024  0.062   3.269
               golden_cross    3101.0  -0.031   -0.101  -0.001 -0.099  -2.210
               kd_cross       33969.0   0.015    0.080  -0.014  0.094   6.977
               macd_cross     12413.0   0.061    0.165   0.016  0.149   6.632
               ALL           103097.0   0.023    0.061   0.006  0.054   6.938
clean 2024-10+ breakout        2946.0   0.008    0.026   0.000  0.026   0.530
               pullback        6174.0   0.050    0.052   0.049  0.003   0.077
               reversal        4713.0   0.025    0.320  -0.104  0.424  11.873
               golden_cross     765.0  -0.059   -0.024  -0.077  0.053   0.598
               kd_cross        8845.0  -0.042    0.065  -0.088  

7d. What the model relies on. Permutation importance on the lastest test year: how much AUC drops when one feature is shuffled. Unlike LightGBM's built-in gain importance, this is not fooled by high-cardinality or correlated features. If mkt_ret_20 / mkt_vol_20 sit at the top.

In [48]:
from sklearn.inspection import permutation_importance

last_year = max(models)
m = models[last_year]
mask = scored & (years == last_year)
pi = permutation_importance(m, data.loc[mask, FEATURES], data.loc[mask, 'win'],
                            scoring='roc_auc', n_repeats=5, random_state=0, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index= FEATURES).sort_values(ascending= False)
print(f"Permutation importance (AUC drop), test year {last_year}: ")
print(imp.round(4).to_string())




Permutation importance (AUC drop), test year 2026: 
mkt_vol_20       0.0438
mkt_ret_20       0.0311
dist_ma20        0.0000
ma20_slope       0.0000
clv              0.0000
adx_rk           0.0000
days_hi252_rk    0.0000
vol_ratio_rk     0.0000
r20_sec          0.0000
r60_sec          0.0000
atr_pct_rk       0.0000
squeeze_rk       0.0000
log_dvol_rk      0.0000
is_breakout      0.0000
is_pullback      0.0000


7e. Within-year comparison. Table 10c pooled all eight test years before taking the top 30%. Because the model's probabilities move with market volatility, the 'top 30%' was mostly trades from calm years, which were good years anyway. So 10c mixes 'trade selection' with 'market timing'. Here the top 30% is chosen 'inside each year' and the results are then pooled. A calm year can no longer 'beat' a wild year; what is left is how well the model ranks trades within the same year. The year-by-year table for the reversal rule shows whether the effect is consistent or driven by one or two years. 

In [49]:
def within_year_lift(col, q= 0.7):
    t = data[scored].copy()
    t['yr'] = t.index.get_level_values('date').year
    t['top'] = t.groupby('yr')[col].transform(lambda s: s >= s.quantile(q))
    clean = t.index.get_level_values('date') >= CLEAN_START
    
    rows = {}
    for period, mask in [('test 2019+', t['yr'] >= TEST_YEARS[0]), ('clean 2024-10+', clean)]:
        for rule in list(rules) + ['ALL']:
            sel = mask if rule == 'ALL' else mask & (t['rule'] == rule)
            a, b = t.loc[sel & t['top'], 'R'], t.loc[sel & ~t['top'], 'R']
            rows[(period, rule)] = pd.Series({
                'n': sel.sum(), 'R_top30': a.mean(), 'R_rest': b.mean(),
                'lift': a.mean() - b.mean(),
                't': ttest_ind(a, b, equal_var=False).statistic
            })
            
    return pd.DataFrame(rows).T.round(3)

def by_year(col, rule, q=0.7):
    t = data[scored & (data['rule'] == rule)].copy()
    t['yr'] = t.index.get_level_values('date').year
    t['top'] = t.groupby('yr')[col].transform(lambda s: s >= s.quantile(q))
    g = t.groupby('yr')
    return pd.DataFrame({'n': g.size(),
                         'R_top30': g.apply(lambda x: x.loc[x['top'], 'R'].mean()),
                         'R_rest': g.apply(lambda x: x.loc[~x['top'], 'R'].mean())}).round(3)
    
print("LightGBM, top 30% chosen within each year:")
print(within_year_lift('p_lgb').to_string())
print("\nLogistic, top 30% chosen within each year:")
print(within_year_lift('p_log').to_string())
print("\nReversal rule, LightGBM, year by year:")
print(by_year('p_lgb', 'reversal').to_string())

LightGBM, top 30% chosen within each year:
                                    n  R_top30  R_rest   lift       t
test 2019+     breakout       11934.0   -0.016  -0.025  0.009   0.369
               pullback       23789.0    0.136  -0.015  0.151   8.936
               reversal       17891.0    0.174  -0.041  0.215  12.112
               golden_cross    3101.0   -0.010  -0.040  0.029   0.643
               kd_cross       33969.0    0.128  -0.053  0.181  14.054
               macd_cross     12413.0    0.227  -0.049  0.276  13.152
               ALL           103097.0    0.135  -0.037  0.172  22.652
clean 2024-10+ breakout        2946.0   -0.042   0.022 -0.065  -1.294
               pullback        6174.0    0.085   0.038  0.047   1.373
               reversal        4713.0    0.251  -0.143  0.394  12.025
               golden_cross     765.0   -0.072  -0.054 -0.018  -0.182
               kd_cross        8845.0    0.118  -0.131  0.249   9.871
               macd_cross      3344.0    0.233 

## 8. Ablation and Robustness Tests

8a. Retrain without the two market-regime features. Step 10d said the model leans almost entirely on `mkt_vol_20` and `mkt_ret_20` and re-run the same walk-forward. Step 9 is wrapped in a function so a feature set can be swapped in on line.

In [50]:

def walk_forward(features, tag):
    pl, pg = f'p_lgb_{tag}', f'p_log_{tag}'
    data[pl], data[pg] = np.nan, np.nan
    for year in TEST_YEARS:
        test = (years == year)
        if not test.any():
            continue
        cut   = close.index[pos_d[dates[test].min()] - EMBARGO]
        train = (dates <= cut)
        valid = train & (years == year - 1)
        fit   = train & ~valid

        m = lgb.LGBMClassifier(**LGB)
        m.fit(data.loc[fit, features], data.loc[fit, 'win'],
              eval_X=data.loc[valid, features], eval_y=data.loc[valid, 'win'],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        data.loc[test, pl] = m.predict_proba(data.loc[test, features])[:, 1]

        lr = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                           LogisticRegression(C=0.1, max_iter=1000))
        lr.fit(data.loc[train, features], data.loc[train, 'win'])
        data.loc[test, pg] = lr.predict_proba(data.loc[test, features])[:, 1]
        print(f"{year}: trees {m.best_iteration_:>4}")

FEATURES_NOMKT = [f for f in FEATURES if not f.startswith('mkt_')]
walk_forward(FEATURES_NOMKT, 'nomkt')

def auc_rows(pl, pg):
    def block(mask):
        sub = data[mask & scored]
        return pd.Series({'n': len(sub),
                          'AUC lgb': roc_auc_score(sub['win'], sub[pl]),
                          'AUC logistic': roc_auc_score(sub['win'], sub[pg])})
    rows = {str(y): block(years == y) for y in TEST_YEARS if (years == y).any()}
    rows['all test years'] = block(years >= TEST_YEARS[0])
    rows['clean 2024-10+'] = block(dates >= CLEAN_START)
    return pd.DataFrame(rows).T.round(3)

print("\nAUC without market features:")
print(auc_rows('p_lgb_nomkt', 'p_log_nomkt').to_string())
print("\nWithin-year lift without market features (LightGBM):")
print(within_year_lift('p_lgb_nomkt').to_string())
print("\nWithin-year lift without market features (logistic):")
print(within_year_lift('p_log_nomkt').to_string())

2019: trees   13
2020: trees   43
2021: trees    1
2022: trees    6
2023: trees    2
2024: trees    3
2025: trees  165
2026: trees    1

AUC without market features:
                       n  AUC lgb  AUC logistic
2019             12413.0    0.559         0.555
2020             13136.0    0.483         0.502
2021             13488.0    0.528         0.540
2022             13794.0    0.529         0.549
2023             13274.0    0.503         0.503
2024             13937.0    0.514         0.533
2025             13464.0    0.499         0.494
2026              9591.0    0.495         0.494
all test years  103097.0    0.497         0.516
clean 2024-10+   26787.0    0.492         0.496

Within-year lift without market features (LightGBM):
                                    n  R_top30  R_rest   lift      t
test 2019+     breakout       11934.0    0.064  -0.030  0.094  2.372
               pullback       23789.0   -0.011   0.038 -0.050 -2.284
               reversal       17891.0    0.05

8b. Reversal trades across market-volatility buckets. If the model's only useful input is market volatility, a lookup table should so the same job without a model. Trades are bucketed by `mkt_vol_20` and the average R of reversal trades is shown per bucket, overall and by year.

In [51]:
pre = dates < pd.Timestamp(f'{TEST_YEARS[0]}-01-01')
edges = data.loc[pre, 'mkt_vol_20'].quantile([0, .2, .4, .6, .8, 1]).to_numpy().copy()
edges[0], edges[-1] = -np.inf, np.inf
data['vol_q'] = pd.cut(data['mkt_vol_20'], edges, labels=[1,2,3,4,5])
print("mkt_vol_20 bucket edges: ", np.round(edges[1:-1], 4))

rv = data[(data['rule'] == 'reversal') & (years >= TEST_YEARS[0])]
print('\n Reversal, avg R by market-vol bucket (1 = calmest): ')
print(rv.groupby('vol_q', observed= True)['R'].agg(['size', 'mean']).round(3).to_string())
print('\n Reversal, avg R by year x bucket: ')
print(rv.pivot_table(index= rv.index.get_level_values('date').year, columns= 'vol_q', values= 'R', aggfunc= 'mean', observed= True).round(2).to_string())

mkt_vol_20 bucket edges:  [0.0054 0.0063 0.0074 0.0108]

 Reversal, avg R by market-vol bucket (1 = calmest): 
       size   mean
vol_q             
1       294  0.022
2       437 -0.106
3       907 -0.092
4      6855  0.002
5      9016  0.088

 Reversal, avg R by year x bucket: 
vol_q     1     2     3     4     5
date                               
2019   0.42 -0.16 -0.13  0.16  0.45
2020  -0.25  0.00 -0.43 -0.13  0.08
2021    NaN   NaN  0.07  0.00  0.11
2022    NaN   NaN   NaN   NaN -0.06
2023    NaN   NaN -0.33 -0.13  0.20
2024    NaN  0.57  0.09  0.09  0.51
2025    NaN -0.54 -0.30  0.09  0.16
2026    NaN   NaN -0.27 -0.18 -0.16


## 9. Results and Discussion

### Main Findings

|Finding| Evidence| Interpretation|
|--|--|--|
|Overall discrimination was weak| LightGBM AUC: 0.512, Logistic AUC: 0.53 | Neither model demonstrated strong general out of sample classification ability|
|Results varied substantially by year| Annual LightGBM AUC ranged from approximately 0.51 to 0.57| Predictive relationship were not stable across market regimes|
| Predicted probabilities were poorly calibrated | Win rate and average R did not increse consistently across probanility deciles | Model probabilities should not be used directly for expected value decisions without calibration. |
| LightGBM depended mainly on market conditions | mkt_vol_20 and mkt_ret_20 dominiated 2026 permutation importance | The model primarily learned market timing rather than stock specific selection.|
| Removing market features eliminated most performance | Post-October 2024 LightGBM AUC fell to 0.492 | The remaining technical and cross sectional features did not generalise reliably.|
| Some rule-level lifts remained | Reversal, KD, and MACD candidates showed positive lift in selected tests | These effects require additional validation because they may reflect market-regime exposure or repeated testing.




## 10. Limitations

This study has several important limitations:

1. Survivorship bias: Data before October 2024 may not contain complete histories for securities that subsequently delisted. Even the later period should be described as having improved coverage unless completeness has been independently verified.

2. Delisting treatment: Positions are closed at the last available price when observations disappear. This may overstate returns when shareholders receive less than the final quoted price.

3. Historical classifications: Security type and sector are obtained from the lastest availble reference record rather than a fully point-in-time classification database.

4. Overlapping observations: Trades overlap in time and share exposure to common market events. Therefore, ordinary Welch t-statistics may include data-mined effects.

5. Multiple testing: Six rules, several models, multiple periods and different selection thresholds were compared. Reported results may include data-mined effects.

6. Transaction costs: Costs are estimated using liquidity tiers and do not model order size, bid-ask spread, market impact or changing market conditions.

7. Np portfolio constraints: Trade level average R does not account for limited capital, simultaneous positions, position sizing or portfolio level drawdowns.

8. Probability calibration: The model probabilities are not formally calibrated, making the expected-value calculation unreliable.

9. Incomplete 2026 period: Results for 2026 cover only part of the calender year and should not be compared directly with full-year results.

## Conclusion

This experiment finds limited evidence that machine learning improves technical rule trade selection. LightGBM occasionally produced useful rankings for reversal, KD, and MACD candidates, but its overall discrimination was weak and most of its measurable signal came from broad market regime variables.

After removing market return and volatility, performance fell to approximately random classification in the high confidence data period. The current evidence therefore supports using market regime analysis as a seperate strategy filter, but does not yet justify deploying the machine learing model as a stock selection system.

Future work could incorporate additional information, such as sector classifications, financial metrics, company reports, and news sentiment, to improve the models’ predictive performance and robustness.